In [1]:
# Cell 1: Install and Download Data
!pip install -q torch nltk

import nltk
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import matplotlib.pyplot as plt
import torch


from torch.utils.data import Dataset, DataLoader
from collections import Counter

# Download brown corpus without long print outputs
nltk.download('brown', quiet=True)
from nltk.corpus import brown

print("Environment setup complete. PyTorch version:", torch.__version__)


Environment setup complete. PyTorch version: 2.11.0+cu130


In [2]:
# Cell 2: Dataset Generation
class Bengio2003Dataset(Dataset):
    def __init__(self, context_size=5, max_vocab_size=10000):
        self.context_size = context_size

        # 1. Gather and lowercase text tokens
        raw_words = [word.lower() for word in brown.words()]

        # 2. Build vocabulary mapping
        word_counts = Counter(raw_words)
        self.word_to_idx = {"<UNK>": 0}

        most_common = word_counts.most_common(max_vocab_size - 1)
        for word, _ in most_common:
            self.word_to_idx[word] = len(self.word_to_idx)

        self.idx_to_word = {idx: word for word, idx in self.word_to_idx.items()}
        self.vocab_size = len(self.word_to_idx)

        # 3. Vectorize text tokens
        indexed_words = [self.word_to_idx.get(word, self.word_to_idx["<UNK>"]) for word in raw_words]

        # 4. Extract context windows (X) and next-word targets (Y)
        self.X = []
        self.Y = []
        for i in range(len(indexed_words) - self.context_size):
            self.X.append(indexed_words[i : i + self.context_size])
            self.Y.append(indexed_words[i + self.context_size])

        self.X = torch.tensor(self.X, dtype=torch.long)
        self.Y = torch.tensor(self.Y, dtype=torch.long)

        print(f"Dataset Built! Total Vocab: {self.vocab_size} | Total Windows: {len(self.X)}")

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]


In [27]:
# Cell 4: Sanity Check Verification
# Setup hardware accelerator
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running execution pipeline on: {device}")

# Hyperparameters
CONTEXT_SIZE = 5      # (n-1) context words
VOCAB_SIZE = 5000     # Limit vocab size for faster execution
BATCH_SIZE = 64

# Instantiate pipeline components
dataset = Bengio2003Dataset(context_size=CONTEXT_SIZE, max_vocab_size=VOCAB_SIZE)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

# Grab a single sample batch
x_sample, y_sample = next(iter(dataloader))
x_sample, y_sample = x_sample.to(device), y_sample.to(device)

print("\n--- Pipeline Diagnostic Shapes ---")
print("Input Batch Matrix X shape :", x_sample.shape)  # Expected: [64, 5]
print("Target Label Vector Y shape:", y_sample.shape)  # Expected: [64]
print("Pipeline Successfully Verified! ✅")

print(x_sample.shape, y_sample.shape)
print(len(dataset))




Running execution pipeline on: cuda
Dataset Built! Total Vocab: 5000 | Total Windows: 1161187

--- Pipeline Diagnostic Shapes ---
Input Batch Matrix X shape : torch.Size([64, 5])
Target Label Vector Y shape: torch.Size([64])
Pipeline Successfully Verified! ✅
torch.Size([64, 5]) torch.Size([64])
1161187


In [28]:
## Sample and print few words
raw_words = [word.lower() for word in brown.words()]
for word in x_sample:
    arr = word.flatten().tolist()
    print(' '.join([dataset.idx_to_word[idx] for idx in arr]))


and got a cool <UNK>
dull <UNK> , like metal
. finally , <UNK> <UNK>
comparison with the heat energy
top of <UNK> , from
law by promising to build
all children in the same
town what happened to anyone
took the seat at his
to items whose right of
<UNK> adjusted her eyes to
, where he enjoyed the
only part-time , in the
citizens group of johnston will
clark of <UNK> , <UNK>
riding the range in the
bank <UNK> , the <UNK>
and water <UNK> . the
other <UNK> of the contribution
expenses are not covered by
signs pointing the way there
, four <UNK> , kept
out the door . ``
society , that he rather
. a struggle that is
of such materials in the
be due in large part
consisted of <UNK> the sections
substantial <UNK> to be made
we were after all making
sighed heavily . `` trouble
more classical <UNK> problem :
always felt that he could
an angry <UNK> , their
and <UNK> and less evident
fall <UNK> . spectacular displays
who <UNK> it and is
. the <UNK> had died
had a good health program
the style of ol

In [29]:
## Create datasets

#Training set
TRAINING_BATCH_SIZE = 800000
trainig_dataloader = DataLoader(dataset, batch_size=TRAINING_BATCH_SIZE, shuffle=True)
X_train, Y_train = next(iter(trainig_dataloader))
print(X_train.shape, Y_train.shape)

#Validation set
DEV_BATCH_SIZE = 200000
dev_dataloader = DataLoader(dataset, batch_size=DEV_BATCH_SIZE, shuffle=True)
X_val, Y_val = next(iter(dev_dataloader))
print(X_val.shape, Y_val.shape)

#Test set
TEST_BATCH_SIZE = 20000
test_dataloader = DataLoader(dataset, batch_size=TEST_BATCH_SIZE, shuffle=True)
X_test, Y_test = next(iter(test_dataloader))
print(X_test.shape, Y_test.shape)

torch.Size([800000, 5]) torch.Size([800000])
torch.Size([200000, 5]) torch.Size([200000])


In [30]:
## Initialize neural network
g = torch.Generator().manual_seed(2147483647)
block_size = 5
embedding_dimension = 10
hidden_layer_size = 100

C = torch.randn((VOCAB_SIZE, embedding_dimension), generator=g)
W1 = torch.randn((embedding_dimension * block_size, hidden_layer_size), generator=g)
b1 = torch.randn(hidden_layer_size, generator=g)
W2 = torch.randn((hidden_layer_size, VOCAB_SIZE), generator=g)
b2 = torch.randn(VOCAB_SIZE, generator=g)

mat_dimension = block_size * embedding_dimension

embeddings = C[X_train]
h = torch.tanh(embeddings.view(-1, mat_dimension) @ W1 + b1)

parameters = [C, W1, b1, W2, b2]
num_params = [sum(p.nelement() for p in parameters)]
print(num_params)
for p in parameters:
  p.requires_grad = True

print(W1.shape, b1.shape, W2.shape, b2.shape)

[560100]
torch.Size([50, 100]) torch.Size([100]) torch.Size([100, 5000]) torch.Size([5000])


In [35]:
## Train neural network
sample_size = 50
step = []
loss_values = []
for i in range(1000000):
  ## sample random entries from dataset
  ix = torch.randint(0, X_train.shape[0], (sample_size,))

  emb = C[X_train[ix]]
  h = torch.tanh(emb.view(-1, mat_dimension) @ W1 + b1)
  logits = h @ W2 + b2
  loss = F.cross_entropy(logits, Y_train[ix])
  if i % 10000 == 0:
   print("Iteration " + str(i) +  " Loss: " + str(loss.item()))

  for p in parameters:
    p.grad = None
  loss.backward()

  learning_rate = 0.1 if i < 500000 else 0.01

  for p in parameters:
    p.data += -learning_rate * p.grad

  step.append(i)
  loss_values.append(loss.log10().item())

Iteration 0 Loss: 6.499993801116943
Iteration 10000 Loss: 6.395699977874756
Iteration 20000 Loss: 6.0868964195251465
Iteration 30000 Loss: 6.980344295501709
Iteration 40000 Loss: 5.651482105255127
Iteration 50000 Loss: 5.1975626945495605
Iteration 60000 Loss: 6.079570293426514
Iteration 70000 Loss: 6.129611015319824
Iteration 80000 Loss: 5.971522331237793
Iteration 90000 Loss: 5.621184825897217
Iteration 100000 Loss: 5.723826885223389
Iteration 110000 Loss: 4.81234884262085
Iteration 120000 Loss: 6.1022467613220215
Iteration 130000 Loss: 5.8552045822143555
Iteration 140000 Loss: 6.5180253982543945
Iteration 150000 Loss: 5.694355487823486
Iteration 160000 Loss: 5.997671127319336
Iteration 170000 Loss: 5.021482467651367
Iteration 180000 Loss: 5.648169994354248
Iteration 190000 Loss: 5.876833438873291
Iteration 200000 Loss: 5.902097702026367
Iteration 210000 Loss: 5.697790145874023
Iteration 220000 Loss: 6.359250068664551
Iteration 230000 Loss: 5.5439653396606445
Iteration 240000 Loss: 5.

In [36]:
emb = C[X_val]
h = torch.tanh(emb.view(-1, mat_dimension) @ W1 + b1)
logits = h @ W2 + b2
loss = F.cross_entropy(logits, Y_val)
print(loss.item())

5.738745212554932


In [37]:
emb = C[X_test]
h = torch.tanh(emb.view(-1, mat_dimension) @ W1 + b1)
logits = h @ W2 + b2
loss = F.cross_entropy(logits, Y_test)
print(loss.item())

5.693081378936768


In [53]:
g = torch.Generator().manual_seed(2147783647)
context = [0] * block_size
paragraph = []
sentence = []
for i in range(500):
  embedding = C[torch.tensor([context])]
  h = torch.tanh(embedding.view(1, -1) @ W1 + b1)
  logits = h @ W2 + b2
  probs = F.softmax(logits, dim=1)
  ix = torch.multinomial(probs, num_samples=1, replacement=True, generator=g).item()
  word = dataset.idx_to_word[ix]
  sentence.append(word)
  context = context[1:] + [ix]
  if word == ".":
    sentence = ' '.join(sentence)
    paragraph.append(sentence)
    sentence = []


print('\n'.join(paragraph))

when territory the .
to which the starting rather .
<UNK> whether the rather have about sitting the of prices and he .
and of .
sacrifice ) time fine for be '' of <UNK> of <UNK> cut the has free strength who the , , men no whole ; and an <UNK> 2 he had county in on god against his in a , took the the to such friend <UNK> proper it the many a it the will , goes could that at his smart the <UNK> quarter know also used on or is for , of a , of full-time .
themselves 1956 ( and revolutionary .
the must <UNK> to to functional on the <UNK> i the he and young the performance larger .
from ? , systems for these i as had visual the <UNK> the ! .
went case louisiana <UNK> further so into <UNK> <UNK> those the in make <UNK> a said churches <UNK> <UNK> <UNK> <UNK> over , <UNK> is , forest , , , never .
with , yesterday churches islands the ? of .
he places existed <UNK> that and his of <UNK> to found <UNK> <UNK> <UNK> was brown those feature in his using tight <UNK> government with of for rates ne